In [583]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,KFold,cross_val_score
from sklearn.linear_model import LinearRegression,Ridge,Lasso
import seaborn as sns
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.metrics import r2_score,mean_absolute_error
from sklearn.decomposition import  PCA
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor,RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from xgboost import XGBClassifier
import category_encoders  as ce


In [584]:
!pip install category_encoders

In [585]:
df=pd.read_csv("/content/selection").drop(columns=['pooja room',"others",'Unnamed: 0','study room','floorNum'])
df.shape


(3234, 12)

In [586]:
!pip install xgboost


In [587]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_score
0,flat,sector 36,0.82,3.0,2.0,2,New Property,908.0,0.0,0.0,0.0,Low
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1432.0,1.0,0.0,0.0,Low
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,0.0,Low
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,1.0,High
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,630.0,0.0,1.0,0.0,High


In [588]:
df.columns

Index(['property_type', 'sector', 'price', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_score'],
      dtype='object')

In [589]:
x=df.drop("price",axis=1)
y=df["price"]

In [590]:
y_trans=np.log1p(y)

In [591]:
columns=['property_type', 'sector', 'balcony','agePossession',
       'luxury_score','furnishing_type']

In [592]:
def categorize(x):
  if x==0:
    return "unfurnished"
  elif x==1:
    return 'semiunfurnished'
  else:
    return 'furnished'

In [593]:
df["furnishing_type"]=df["furnishing_type"].apply(categorize)
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3234 entries, 0 to 3233
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   property_type    3234 non-null   object 
 1   sector           3234 non-null   object 
 2   price            3234 non-null   float64
 3   bedRoom          3234 non-null   float64
 4   bathroom         3234 non-null   float64
 5   balcony          3234 non-null   object 
 6   agePossession    3234 non-null   object 
 7   built_up_area    3234 non-null   float64
 8   servant room     3234 non-null   float64
 9   store room       3234 non-null   float64
 10  furnishing_type  3234 non-null   object 
 11  luxury_score     3234 non-null   object 
dtypes: float64(6), object(6)
memory usage: 303.3+ KB


In [594]:
 preprocessor=ColumnTransformer([('num',StandardScaler(),['bedRoom','built_up_area','servant room','store room','bathroom']),
                                ('cat',OneHotEncoder(handle_unknown='ignore',drop="first"),['agePossession']),
                                  ('order',OrdinalEncoder(),['balcony','luxury_score','property_type','furnishing_type']),('target',ce.TargetEncoder(),["sector"])
                                  ],remainder="passthrough")
pipeline=Pipeline([('prerocessor',preprocessor),('regressor',LinearRegression())])


In [595]:
kfold=KFold(n_splits=10,shuffle=True,random_state=42)
score=cross_val_score(pipeline,x,y_trans,cv=kfold,scoring="r2")

In [614]:
print(score.mean(),score.std())

0.7925487274111884 0.02351619814880503


In [597]:
x_train,x_test,y_train,y_test=train_test_split(x,y_trans,test_size=0.2,random_state=42)

In [598]:
pipeline.fit(x_train,y_train)

Pipeline(steps=[('prerocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'built_up_area',
                                                   'servant room', 'store room',
                                                   'bathroom']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['agePossession']),
                                                 ('order', OrdinalEncoder(),
                                                  ['balcony', 'luxury_score',
                                                   'property_type',
                                                   'furnishing_type']),
                                                 ('target', TargetEncoder(),
                                                  ['sector'])])),
                ('regressor', LinearRegression())])

In [599]:
y_pred=pipeline.predict(x_test)
y_pred=np.expm1(y_pred)

In [600]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.4371342885048375

In [601]:
def scorer(model_name,model):
  output=[]
  output.append(model_name)
  pipeline=Pipeline([('prerocessor',preprocessor),(model_name,model)])
  kfold=KFold(n_splits=10,shuffle=True,random_state=42)
  score=cross_val_score(pipeline,x,y_trans,cv=kfold,scoring="r2")
  output.append(score.mean())
  x_train,x_test,y_train,y_test=train_test_split(x,y_trans,test_size=0.2,random_state=42)
  pipeline.fit(x_train,y_train)
  y_pred=pipeline.predict(x_test)
  y_pred=np.expm1(y_pred)
  output.append(mean_absolute_error(np.expm1(y_test),y_pred))
  return output



In [602]:
model_dict={
    "linear_model":LinearRegression(),
    'decision_tree':DecisionTreeRegressor(),
    'svm':SVR(kernel='rbf'),
    'random_forest':RandomForestRegressor(),
    'gradient_boosting':GradientBoostingRegressor(),
    'ridge':Ridge(),
    'lasso':Lasso(),
    'xgboost':XGBRegressor(),
    'mlp': MLPRegressor(),
    'extra_tree':ExtraTreesRegressor()

}

In [603]:
model_output=[]
for model_name,model in model_dict.items():
  model_output.append(scorer(model_name,model))

In [604]:
model_df=pd.DataFrame(model_output,columns=["model_name","r2_score","mean_score"]).sort_values("mean_score",ascending=True)


In [605]:
model_df

,model_name,r2_score,mean_score
9,extra_tree,0.876008,0.291967
3,random_forest,0.872856,0.298066
7,xgboost,0.878216,0.298130
4,gradient_boosting,0.855005,0.334950
2,svm,0.822617,0.371275
1,decision_tree,0.761895,0.391386
8,mlp,0.813107,0.397591
0,linear_model,0.792549,0.437134
5,ridge,0.792590,0.437579
6,lasso,-0.002016,0.877895


In [606]:
def scorer1(model_name,model):
  output1=[]
  output1.append(model_name)
  preprocessor=ColumnTransformer([('num',StandardScaler(),['bedRoom','built_up_area','servant room','store room','bathroom']),
                                ('cat',OneHotEncoder(handle_unknown='ignore',drop="first"),["sector",'agePossession']),
                                  ('order',OrdinalEncoder(),['balcony','luxury_score','property_type','furnishing_type'])
                                  ],remainder="passthrough")
  pipeline=Pipeline([('prerocessor',preprocessor),('pca',PCA(n_components=45)),(model_name,model)])
  kfold=KFold(n_splits=10,shuffle=True,random_state=42)
  score=cross_val_score(pipeline,x,y_trans,cv=kfold,scoring="r2")
  output1.append(score.mean())
  x_train,x_test,y_train,y_test=train_test_split(x,y_trans,test_size=0.2,random_state=42)
  pipeline.fit(x_train,y_train)
  y_pred=pipeline.predict(x_test)
  y_pred=np.expm1(y_pred)
  output1.append(mean_absolute_error(np.expm1(y_test),y_pred))
  return output1



In [607]:
model_output1=[]
for model_name,model in model_dict.items():
  model_output1.append(scorer1(model_name,model))


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categ

In [608]:
model_df1=pd.DataFrame(model_output,columns=["model_name","r2_score","mean_score"]).sort_values("mean_score",ascending=True)


In [609]:
model_df1

,model_name,r2_score,mean_score
9,extra_tree,0.876008,0.291967
3,random_forest,0.872856,0.298066
7,xgboost,0.878216,0.298130
4,gradient_boosting,0.855005,0.334950
2,svm,0.822617,0.371275
1,decision_tree,0.761895,0.391386
8,mlp,0.813107,0.397591
0,linear_model,0.792549,0.437134
5,ridge,0.792590,0.437579
6,lasso,-0.002016,0.877895


Target_encoders


In [610]:
def target(model_name,model):
  output1=[]
  output1.append(model_name)
  preprocessor=ColumnTransformer([('num',StandardScaler(),['bedRoom','built_up_area','servant room','store room','bathroom']),
                                ('cat',OneHotEncoder(handle_unknown='ignore',drop="first"),['agePossession']),
                                  ('order',OrdinalEncoder(),['balcony','luxury_score','property_type','furnishing_type']),('target',ce.TargetEncoder(),["sector"])
                                  ],remainder="passthrough")
  pipeline=Pipeline([('prerocessor',preprocessor),(model_name,model)])
  kfold=KFold(n_splits=10,shuffle=True,random_state=42)
  score=cross_val_score(pipeline,x,y_trans,cv=kfold,scoring="r2")
  output1.append(score.mean())
  x_train,x_test,y_train,y_test=train_test_split(x,y_trans,test_size=0.2,random_state=42)
  pipeline.fit(x_train,y_train)
  y_pred=pipeline.predict(x_test)
  y_pred=np.expm1(y_pred)
  output1.append(mean_absolute_error(np.expm1(y_test),y_pred))
  return output1



In [611]:
target_list=[]
for model_name,model in model_dict.items():
  target_list.append(target(model_name,model))


In [612]:
target1=pd.DataFrame(target_list,columns=["model_name","r2_score","mean_score"]).sort_values("mean_score",ascending=True)

In [613]:
target1

,model_name,r2_score,mean_score
9,extra_tree,0.875815,0.286001
3,random_forest,0.872716,0.297078
7,xgboost,0.878216,0.298130
4,gradient_boosting,0.855334,0.334312
2,svm,0.822617,0.371275
1,decision_tree,0.768247,0.379626
8,mlp,0.813495,0.396385
0,linear_model,0.792549,0.437134
5,ridge,0.792590,0.437579
6,lasso,-0.002016,0.877895
